# 01_ingest_bronze.py

## Purpose
Ingest raw CSV files from the landing Volume into a Bronze Delta table using Auto Loader.

## Source
- Volume: `/Volumes/fifa/default/fifa_landing/landing/`
- Format: CSV files named `players_YYYYMMDD.csv`
- All editions (FC 26, FC 27, etc.) are placed in the same landing folder

## Strategy
- Auto Loader in directory listing mode
- Schema evolution: `addNewColumns`
- Trigger: `availableNow=True` (batch mode)
- Idempotent via checkpoint
- Processed files are moved to `archive/` after a 1-day retention period

## Captured Metadata
- `_source_file_path`: full path of the source CSV
- `_source_file_name`: file name of the source CSV
- `_source_mod_time`: file modification timestamp
- `_ingestion_timestamp`: timestamp when the row entered the pipeline

## Output
- Table: `fifa.bronze.ea_players_raw`

## Folder Structure
/Volumes/fifa/default/fifa_landing/

├── landing/ ← CSV de entrada (Auto Loader watches here)

├── archive/ ← CSV procesados (moved by cleanSource)

├── _schema/fifa_players/ ← Inferred schema

└── _checkpoint/fifa_players/ ← Stream state

## Notes
- Checkpoint and schema are stored in the Volume (Free Edition)
- `cloudFiles.cleanSource = MOVE` moves processed files to `archive/` after the retention period
- `awaitTermination()` ensures the notebook waits for the stream to finish

## Why Auto Loader instead of `COPY INTO` or `CTAS`?

| Feature | **Auto Loader** | `COPY INTO` | `CTAS` (`CREATE TABLE AS SELECT`) |
|---|---|---|---|
| **New file detection** | Automatic and incremental (notifies the cloud file listing service) | Manual: you must specify files or paths each time | Does not detect files; loads only once |
| **Execution mode** | Streaming (`readStream`), can run in batches with `trigger(availableNow=True)` | One-off batch | One-off batch |
| **Schema inference & evolution** | Yes: `schemaLocation` + `schemaEvolutionMode` detects and evolves the schema automatically | Requires a fixed or manually defined schema | Schema is inferred once at table creation |
| **Idempotency / re-execution** | Manages state with `checkpointLocation`; does not reprocess already-loaded files | Re-running can duplicate data if not controlled | Re-running recreates the table from scratch |
| **File metadata** | `_metadata.file_path`, `_metadata.file_name`, `_metadata.file_modification_time` available | Limited | Not available directly |
| **Scalability (millions of files)** | Designed for it (efficient incremental listing) | Degrades with many files | N/A |
| **Ideal for** | Continuous or recurring batch ingestion of growing data | One-off or sporadic loading of small files | Creating a table from a one-off query |

### Summary

We use **Auto Loader** because:

1. **It detects new files automatically** — no need to specify paths on each run.
2. **It evolves the schema** — if new columns appear in the CSV, they are added without breaking the pipeline.
3. **It is idempotent** — thanks to `checkpointLocation`, re-running does not duplicate data.
4. **It captures metadata** — file path, name, and modification date of the source file.
5. **It scales to millions of files** efficiently.

`COPY INTO` is simpler for one-off loads, but it does not handle schema evolution or checkpointing.
`CTAS` only works for creating a table once, not for recurring ingestion.

Auto Loader → table Bronze with metadatos

In [0]:
from pyspark.sql.functions import current_timestamp  # import the current_timestamp function

# --- Configuration parameters ---
catalog_name = "fifa"
volume_name = "fifa_landing"

# --- Fixed paths ---
# Source path: where the CSVs are located. Auto Loader watches this folder.
# Any new file that appears here will be processed when this cell or the pipeline runs.
source_path = f"/Volumes/{catalog_name}/default/{volume_name}/landing"

# Schema location: where Auto Loader stores the inferred schema.
# It records the columns it has seen (name and type). This enables schema evolution:
# if a new CSV brings a different column, Auto Loader detects it and updates this schema.
schema_location = f"/Volumes/{catalog_name}/default/{volume_name}/_schema/fifa_players"

# Checkpoint location: where Auto Loader stores its internal state.
# It records which files have already been processed so they are not read again.
# This guarantees idempotency. It should go in DBFS, but in Free Edition it goes in the Volume.
checkpoint_location = f"/Volumes/{catalog_name}/default/{volume_name}/_checkpoint/fifa_players"


# --- Target table ---
target_table = f"{catalog_name}.bronze.ea_fc_players_raw"

# --- Auto Loader ingestion ---
query = (spark.readStream  # readStream performs an incremental read of the given directory (source_path)
  .format("cloudFiles")  # cloudFiles is the Auto Loader format (not a file format, but an incremental reader)
  .option("cloudFiles.format", "csv")  # actual file format, in this case CSV
  .option("header", "true")  # whether the file has a header row
  .option("cloudFiles.schemaLocation", schema_location)  # path where the inferred schema is stored
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # schema evolution mode
  .option("cloudFiles.cleanSource", "MOVE")  # move processed files to a backup folder
  .option("cloudFiles.cleanSource.moveDestination", f"/Volumes/{catalog_name}/default/{volume_name}/archive/")  # backup folder
  .option("cloudFiles.cleanSource.retentionDuration", "1 day")  # retention time for files before moving
  .load(source_path)  # load files from the given path
  # Add extra columns with source info and ingestion timestamp (required metadata) + all stream records
  .selectExpr(
        "*",
        "_metadata.file_path AS _source_file_path",
        "_metadata.file_name AS _source_file_name",
        "_metadata.file_modification_time AS _source_mod_time"
    )
  .withColumn("_ingestion_timestamp", current_timestamp())  # add ingestion timestamp column
  .writeStream  # write the stream
  .option("checkpointLocation", checkpoint_location)  # path where the stream's internal state is stored
  .option("mergeSchema", "true")  # allows the Delta table schema to evolve when new columns are added
  .trigger(availableNow=True)  # processes everything pending and stops. Does not keep running.
  .toTable(target_table)  # saves the stream to the specified Delta table
)
query.awaitTermination()  # wait for the stream to finish before continuing with the rest of the notebook

LIMIT 10 of the Bronze table. Quick verification that we have entered the raw data into our Bronze table.

In [0]:
%sql
select * from fifa.bronze.ea_fc_players_raw order by `_ingestion_timestamp` desc limit 10 ;